# PVFit Optuna Optimization Analysis

Analysis of NSGA-II multi-objective optimization results.

**Objectives** (4-D Pareto front):
- Efficiency (true-denom) → maximize
- Fake rate (rec-denom) → minimize
- Merge rate (true-denom) → minimize
- Split rate (rec-denom) → minimize

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from itertools import combinations
from pathlib import Path

from pvfit.optimize.constants import OBJ_COLS, OBJ_LABELS, OBJ_DIRS, BASELINE
from pvfit.optimize.backends.pvfit import PARAM_COLS
from pvfit.optimize.analysis import (
    load_study_data, profile_scan, pareto_2d,
    normalize_objectives, utopia_distance,
)

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
})

# ── Load data ──
STUDY_DIR = Path('output/optuna/pvfit_full10_g0')
df, df_unique, df_p = load_study_data(STUDY_DIR)

print(f'Total complete trials: {len(df)}')
print(f'Unique parameter signatures: {len(df_unique)}')
print(f'Unique Pareto front size: {len(df_p)}')
print(f'Generations: {df["generation"].nunique()}')

---
## 1. Pareto Front Visualization

### 1.1 Pair-wise 2D projections (6 plots)

Each plot shows all trials (grey) with Pareto front highlighted (colored by efficiency).
Baseline marked with ★.

In [ ]:
pairs = list(combinations(range(4), 2))
fig, axes = plt.subplots(2, 3, figsize=(20, 13))
axes_flat = axes.ravel()

# Pareto points colored by efficiency
df_np = df_unique[~df_unique['is_pareto']].copy()

for idx, (i, j) in enumerate(pairs):
    ax = axes_flat[idx]
    cx, cy = OBJ_COLS[i], OBJ_COLS[j]

    # All non-pareto trials
    ax.scatter(df_np[cx], df_np[cy], c='#cccccc', s=6, alpha=0.4, zorder=1,
               rasterized=True)

    # Pareto front colored by efficiency
    sc = ax.scatter(df_p[cx], df_p[cy], c=df_p['efficiency_true_overall'],
                    cmap='RdYlGn', s=18, alpha=0.85, edgecolors='k', linewidths=0.3,
                    vmin=0.5, vmax=0.85, zorder=2)

    # Baseline
    ax.scatter(BASELINE[cx], BASELINE[cy], marker='*', c='blue', s=250,
               edgecolors='k', linewidths=1.0, zorder=3)

    ax.set_xlabel(OBJ_LABELS[i])
    ax.set_ylabel(OBJ_LABELS[j])

    # Optimization direction arrows
    dir_text_x = '→ better' if OBJ_DIRS[i] == 'max' else '← better'
    dir_text_y = '↑ better' if OBJ_DIRS[j] == 'max' else '↓ better'
    ax.set_title(f'{OBJ_LABELS[i]}  vs  {OBJ_LABELS[j]}\n({dir_text_x}, {dir_text_y})',
                 fontsize=10, pad=8)

# Shared colorbar on the right, outside the subplot grid
fig.subplots_adjust(left=0.06, right=0.88, top=0.91, bottom=0.08, wspace=0.30, hspace=0.38)
cbar_ax = fig.add_axes([0.90, 0.15, 0.015, 0.70])   # [left, bottom, width, height]
fig.colorbar(sc, cax=cbar_ax, label='Efficiency')

# Legend at the bottom, outside axes area
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#cccccc', markersize=6, label='All trials'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='green', markersize=8,
           markeredgecolor='k', markeredgewidth=0.3, label='Pareto front'),
    Line2D([0], [0], marker='*', color='w', markerfacecolor='blue', markersize=14,
           markeredgecolor='k', markeredgewidth=0.8, label='Baseline'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=11,
           frameon=True, bbox_to_anchor=(0.47, 0.0))

fig.suptitle('Pareto Front: 2D Projections (colored by efficiency)', fontsize=14)
plt.show()

### 1.2 Parallel Coordinates Plot

Each line is one Pareto trial. Objectives are normalized so that 'better' is always up.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Normalize each objective to [0, 1] where 1 = best (using imported function)
norm_data = normalize_objectives(df_p, ref_df=df_p)

# Color by efficiency
eff_norm = norm_data[OBJ_COLS[0]].values
cmap = plt.cm.RdYlGn

x = np.arange(4)
for idx in range(len(norm_data)):
    y = norm_data.iloc[idx].values
    color = cmap(eff_norm[idx])
    ax.plot(x, y, c=color, alpha=0.3, linewidth=0.8)

# Baseline (normalize manually with same ref range)
baseline_norm = []
for col, direction in zip(OBJ_COLS, OBJ_DIRS):
    vmin, vmax = df_p[col].min(), df_p[col].max()
    bv = BASELINE[col]
    if direction == 'max':
        baseline_norm.append((bv - vmin) / (vmax - vmin))
    else:
        baseline_norm.append(1 - (bv - vmin) / (vmax - vmin))
ax.plot(x, baseline_norm, 'b--', linewidth=2.5, marker='*', markersize=12, label='Baseline', zorder=5)

ax.set_xticks(x)
ax.set_xticklabels([f'{l}\n({"↑" if d=="max" else "↓"} better)' for l, d in zip(OBJ_LABELS, OBJ_DIRS)])
ax.set_ylabel('Normalized objective (1 = best)')
ax.set_ylim(-0.05, 1.05)
ax.set_title('Parallel Coordinates: Pareto Front (colored by efficiency)')
ax.legend(loc='lower right')

sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
fig.colorbar(sm, ax=ax, label='Efficiency (normalized)', shrink=0.8)
plt.tight_layout()
plt.show()

### 1.3 Convergence: Pareto Front by Generation

Are later generations still pushing the Pareto front?

In [ ]:
# Show eff vs fake_rate colored by generation (the most informative 2D projection)
fig, axes = plt.subplots(1, 2, figsize=(18, 6.5))

for ax, (yi, yl) in zip(axes, [(1, 'Fake rate (rec)'), (2, 'Merge rate (true)')]):
    sc = ax.scatter(df_p['efficiency_true_overall'], df_p[OBJ_COLS[yi]],
                    c=df_p['generation'].astype(int), cmap='viridis', s=20, alpha=0.7,
                    edgecolors='k', linewidths=0.2)
    ax.scatter(BASELINE['efficiency_true_overall'], BASELINE[OBJ_COLS[yi]],
               marker='*', c='red', s=250, edgecolors='k', linewidths=1.0, zorder=5, label='Baseline')
    ax.set_xlabel('Efficiency (true)')
    ax.set_ylabel(yl)
    ax.set_title(f'Pareto Front colored by generation')
    ax.legend()

fig.subplots_adjust(left=0.06, right=0.88, wspace=0.25)
cbar_ax = fig.add_axes([0.90, 0.15, 0.015, 0.70])
fig.colorbar(sc, cax=cbar_ax, label='Generation')
plt.show()

---
## 2. Parameter Profile Scan

For each parameter, we show the **envelope** (best achievable objective value) as a function of the parameter.
This is analogous to profile likelihood in HEP: fix one parameter, optimize over all others.

Implementation: bin the parameter range, in each bin select the trial with the best objective value.

In [ ]:
# Plot profile scans: 8 params × 4 objectives
# profile_scan() is imported from pvfit.optimize.analysis

fig, axes = plt.subplots(len(PARAM_COLS), len(OBJ_COLS), figsize=(22, 28))

for row, param in enumerate(PARAM_COLS):
    for col_idx, (obj, label, direction) in enumerate(zip(OBJ_COLS, OBJ_LABELS, OBJ_DIRS)):
        ax = axes[row, col_idx]
        centers, best, all_v = profile_scan(df, param, obj, direction, n_bins=12)

        # Scatter all points (tiny, grey)
        ax.scatter(df[param], df[obj], c='#dddddd', s=3, alpha=0.3, zorder=1)

        # Box/violin-like: show spread per bin
        bp_positions = []
        bp_data = []
        for c, av in zip(centers, all_v):
            if len(av) >= 3:
                bp_positions.append(c)
                bp_data.append(av)
        if bp_data:
            widths = (centers[-1] - centers[0]) / len(centers) * 0.6 if len(centers) > 1 else 0.1
            ax.boxplot(bp_data, positions=bp_positions, widths=widths,
                       showfliers=False, patch_artist=True,
                       boxprops=dict(facecolor='lightblue', alpha=0.5),
                       medianprops=dict(color='orange', linewidth=1.5),
                       manage_ticks=False)

        # Profile envelope (best)
        ax.plot(centers, best, 'r-o', linewidth=2, markersize=5, zorder=4, label='Profile (best)')

        # Baseline reference line
        ax.axhline(BASELINE[obj], color='blue', linestyle='--', linewidth=1, alpha=0.7, label='Baseline')

        if row == 0:
            ax.set_title(label, fontsize=11)
        if col_idx == 0:
            ax.set_ylabel(param, fontsize=11, fontweight='bold')
        if row == len(PARAM_COLS) - 1:
            ax.set_xlabel(param)

        if row == 0 and col_idx == 0:
            ax.legend(fontsize=8, loc='lower left')

fig.suptitle('Parameter Profile Scans\n(red = best achievable objective; boxes = spread across trials)',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 2.1 Parameter Profile: Focused View

Profile scan for efficiency only, showing clear parameter sensitivity.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.ravel()

for idx, param in enumerate(PARAM_COLS):
    ax = axes[idx]
    obj = 'efficiency_true_overall'

    centers, best, all_v = profile_scan(df, param, obj, 'max', n_bins=12)

    # All points
    ax.scatter(df[param], df[obj], c='#dddddd', s=5, alpha=0.3, zorder=1)

    # Pareto points
    ax.scatter(df_p[param], df_p[obj], c='green', s=8, alpha=0.4, zorder=2, label='Pareto')

    # Profile
    ax.plot(centers, best, 'r-o', linewidth=2, markersize=5, zorder=4, label='Profile')

    # Median per bin
    medians = [np.median(av) for av in all_v]
    ax.plot(centers, medians, 'b--s', linewidth=1.5, markersize=4, zorder=3, label='Median')

    ax.axhline(BASELINE[obj], color='orange', linestyle=':', linewidth=1.5, alpha=0.8)
    ax.set_xlabel(param)
    ax.set_ylabel('Efficiency')
    ax.set_title(param)
    if idx == 0:
        ax.legend(fontsize=8)

fig.suptitle('Parameter Profile: Efficiency (red = envelope, blue = median, orange = baseline)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 3. Working Point Selection

Five approaches:
1. **Utopia distance**: rank Pareto points by normalized Euclidean distance to the ideal point
2. **Constraint-based**: filter Pareto points satisfying physics constraints, then rank
3. **Interesting extremes**: best on each single objective
4. **Merge-constrained Pareto** (NEW): require merge rate < 0.04, then find the 2D Pareto front of (max eff, min fake) within that subset

In [ ]:
# ── 3.1 Utopia distance ranking ──

df_p = df_p.copy()
df_p['utopia_distance'] = utopia_distance(df_p, ref_df=df).values

# Top 10 by utopia distance
top10 = df_p.nsmallest(10, 'utopia_distance')
print('=== Top 10 Working Points (by Utopia distance) ===')
display_cols = ['trial_number', 'utopia_distance'] + OBJ_COLS + PARAM_COLS + ['derived_m_minTracks']
print(top10[display_cols].to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
# ── 3.2 Constraint-based selection ──
# Require: eff >= baseline, fake <=  baseline

mask = (
    (df_p['efficiency_true_overall'] >= BASELINE['efficiency_true_overall']) &
    (df_p['fake_rate_rec_overall'] <=  BASELINE['fake_rate_rec_overall'])
)
df_constrained = df_p[mask].copy()
print(f'\n=== Pareto points passing constraints: {len(df_constrained)} / {len(df_p)} ===')
print(f'  Constraints: eff >= {BASELINE["efficiency_true_overall"]:.4f}, '
      f'fake <= {2*BASELINE["fake_rate_rec_overall"]:.4f}')

if len(df_constrained) > 0:
    # Rank by utopia distance within constrained set
    top_constrained = df_constrained.nsmallest(10, 'utopia_distance')
    print('\nTop 10 constrained working points:')
    print(top_constrained[display_cols].to_string(index=False, float_format='{:.4f}'.format))
else:
    # Relax constraints
    print('No points pass strict constraints. Relaxing...')
    mask_relaxed = (
        (df_p['efficiency_true_overall'] >= 0.95 * BASELINE['efficiency_true_overall']) &
        (df_p['fake_rate_rec_overall'] <= 3 * BASELINE['fake_rate_rec_overall'])
    )
    df_relaxed = df_p[mask_relaxed].copy()
    print(f'  Relaxed: eff >= {0.95*BASELINE["efficiency_true_overall"]:.4f}, '
          f'fake <= {3*BASELINE["fake_rate_rec_overall"]:.4f}')
    print(f'  Points passing: {len(df_relaxed)}')
    if len(df_relaxed) > 0:
        top_relaxed = df_relaxed.nsmallest(10, 'utopia_distance')
        print(top_relaxed[display_cols].to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
# ── 3.3 Extreme points on each objective ──
print('\n=== Best single-objective points on Pareto front ===')
for col, label, direction in zip(OBJ_COLS, OBJ_LABELS, OBJ_DIRS):
    if direction == 'max':
        best = df_p.loc[df_p[col].idxmax()]
    else:
        best = df_p.loc[df_p[col].idxmin()]
    print(f'\nBest {label} ({direction}): trial {int(best["trial_number"])}')
    for c, l in zip(OBJ_COLS, OBJ_LABELS):
        bl = BASELINE[c]
        ratio = best[c] / bl if bl != 0 else float('inf')
        print(f'  {l}: {best[c]:.5f}  (baseline: {bl:.5f}, ratio: {ratio:.2f}x)')

In [ ]:
# ── 3.4 Visualize selected working points ──

# Collect candidate working points
wp_trials = {}

# Best utopia
best_utopia = df_p.nsmallest(1, 'utopia_distance').iloc[0]
wp_trials['Best Utopia'] = best_utopia

# Best efficiency
wp_trials['Best Eff'] = df_p.loc[df_p['efficiency_true_overall'].idxmax()]

# Best fake rate
wp_trials['Best Fake'] = df_p.loc[df_p['fake_rate_rec_overall'].idxmin()]

# Best constrained (if available)
if len(df_constrained) > 0:
    wp_trials['Best Constrained'] = df_constrained.nsmallest(1, 'utopia_distance').iloc[0]

# Radar chart comparing working points
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

angles = np.linspace(0, 2 * np.pi, 4, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

colors = ['red', 'green', 'purple', 'orange', 'cyan']

for (name, trial), color in zip(wp_trials.items(), colors):
    # Normalize to [0,1] where 1=best
    vals = []
    for col, direction in zip(OBJ_COLS, OBJ_DIRS):
        vmin, vmax = df[col].min(), df[col].max()
        if direction == 'max':
            vals.append((trial[col] - vmin) / (vmax - vmin))
        else:
            vals.append(1 - (trial[col] - vmin) / (vmax - vmin))
    vals += vals[:1]
    ax.plot(angles, vals, 'o-', linewidth=2, label=f"{name} (t{int(trial['trial_number'])})", color=color)
    ax.fill(angles, vals, alpha=0.1, color=color)

# Baseline
bl_vals = []
for col, direction in zip(OBJ_COLS, OBJ_DIRS):
    vmin, vmax = df[col].min(), df[col].max()
    if direction == 'max':
        bl_vals.append((BASELINE[col] - vmin) / (vmax - vmin))
    else:
        bl_vals.append(1 - (BASELINE[col] - vmin) / (vmax - vmin))
bl_vals += bl_vals[:1]
ax.plot(angles, bl_vals, 'b--*', linewidth=2, markersize=10, label='Baseline', zorder=5)

ax.set_xticks(angles[:-1])
ax.set_xticklabels([f'{l}\n(1=best)' for l in OBJ_LABELS])
ax.set_ylim(0, 1.05)
ax.set_title('Working Point Comparison (Radar Chart)\n(before knee point)', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.5 Summary table ──
print('\n' + '='*100)
print('WORKING POINT SUMMARY')
print('='*100)

rows = []
for name, trial in wp_trials.items():
    row = {'Label': name, 'Trial': int(trial['trial_number'])}
    for col, label in zip(OBJ_COLS, OBJ_LABELS):
        row[label] = f"{trial[col]:.5f}"
    for p in PARAM_COLS:
        row[p] = trial[p]
    rows.append(row)

# Add baseline
bl_row = {'Label': 'BASELINE', 'Trial': '-'}
for col, label in zip(OBJ_COLS, OBJ_LABELS):
    bl_row[label] = f"{BASELINE[col]:.5f}"
rows.append(bl_row)

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

### 3.6 Merge-Constrained Working Point Selection

**Strategy**: Fix merge rate < 0.03 (slightly below baseline 0.033), then within that subset find points that maximize efficiency and minimize fake rate simultaneously. This reduces the 4D problem to a 2D Pareto front where the trade-off is clear and a single working point can be chosen.

In [ ]:
# ── 3.6 Merge-constrained selection ──
MERGE_CUT = 0.04

mask_merge = df_p['merge_rate_true_overall'] < MERGE_CUT
df_mc = df_p[mask_merge].copy()
print(f'Pareto points with merge rate < {MERGE_CUT}: {len(df_mc)} / {len(df_p)}')
print(f'  Efficiency range: [{df_mc["efficiency_true_overall"].min():.4f}, '
      f'{df_mc["efficiency_true_overall"].max():.4f}]')
print(f'  Fake rate range:  [{df_mc["fake_rate_rec_overall"].min():.4f}, '
      f'{df_mc["fake_rate_rec_overall"].max():.4f}]')

# Build 2D Pareto front: max eff, min fake (using imported function)
df_mc_pareto = pareto_2d(df_mc, 'efficiency_true_overall', 'fake_rate_rec_overall')
df_mc_pareto = df_mc_pareto.sort_values('efficiency_true_overall', ascending=False)
print(f'\n2D Pareto front (max eff, min fake | merge < {MERGE_CUT}): {len(df_mc_pareto)} points')

# Display the 2D Pareto front
display_cols_mc = (['trial_number'] + OBJ_COLS + PARAM_COLS + ['derived_m_minTracks'])
print(f'\nAll 2D Pareto-optimal points (sorted by efficiency ↓):')
print(df_mc_pareto[display_cols_mc].to_string(index=False, float_format='{:.5f}'.format))

In [ ]:
# ── 3.7 Visualization: merge-constrained 2D Pareto front ──

fig, axes = plt.subplots(1, 2, figsize=(20, 7.5))

# ── Left: Eff vs Fake, showing the merge-constrained 2D Pareto front ──
ax = axes[0]

# All Pareto points (grey)
ax.scatter(df_p['efficiency_true_overall'], df_p['fake_rate_rec_overall'],
           c='#dddddd', s=8, alpha=0.4, zorder=1, label='All Pareto')

# Merge-passing points colored by merge rate
sc = ax.scatter(df_mc['efficiency_true_overall'], df_mc['fake_rate_rec_overall'],
                c=df_mc['merge_rate_true_overall'], cmap='coolwarm_r', s=25,
                edgecolors='k', linewidths=0.3, alpha=0.7,
                vmin=0, vmax=MERGE_CUT, zorder=2, label=f'merge < {MERGE_CUT}')

# 2D Pareto front curve
front_sorted = df_mc_pareto.sort_values('efficiency_true_overall')
ax.plot(front_sorted['efficiency_true_overall'], front_sorted['fake_rate_rec_overall'],
        'r-', linewidth=2, alpha=0.8, zorder=3, label='2D Pareto front')
ax.scatter(df_mc_pareto['efficiency_true_overall'], df_mc_pareto['fake_rate_rec_overall'],
           c='red', s=50, edgecolors='k', linewidths=0.8, zorder=4, marker='D')

# Baseline
ax.scatter(BASELINE['efficiency_true_overall'], BASELINE['fake_rate_rec_overall'],
           marker='*', c='blue', s=250, edgecolors='k', linewidths=1.0, zorder=5, label='Baseline')

# Annotate a few key points on the 2D front
for _, row in df_mc_pareto.iterrows():
    ax.annotate(f't{int(row["trial_number"])}',
                (row['efficiency_true_overall'], row['fake_rate_rec_overall']),
                textcoords='offset points', xytext=(5, 5), fontsize=7, alpha=0.7)

ax.set_xlabel('Efficiency (true) → higher better')
ax.set_ylabel('Fake rate (rec) → lower better')
ax.set_title(f'2D Pareto Front (merge < {MERGE_CUT})\nEfficiency vs Fake Rate', fontsize=12)
ax.legend(fontsize=9, loc='upper left')

fig.colorbar(sc, ax=ax, label='Merge rate', shrink=0.8, pad=0.02)

# ── Right: Eff vs Fake zoomed on the interesting region ──
ax2 = axes[1]

# Only merge-constrained subset
ax2.scatter(df_mc['efficiency_true_overall'], df_mc['fake_rate_rec_overall'],
            c=df_mc['split_rate_rec_overall'], cmap='YlOrRd', s=30,
            edgecolors='k', linewidths=0.3, alpha=0.7,
            zorder=2, label=f'merge < {MERGE_CUT} (color=split)')

# 2D Pareto front
ax2.plot(front_sorted['efficiency_true_overall'], front_sorted['fake_rate_rec_overall'],
         'r-', linewidth=2, alpha=0.8, zorder=3)
ax2.scatter(df_mc_pareto['efficiency_true_overall'], df_mc_pareto['fake_rate_rec_overall'],
            c='red', s=60, edgecolors='k', linewidths=0.8, zorder=4, marker='D')

# Baseline
ax2.scatter(BASELINE['efficiency_true_overall'], BASELINE['fake_rate_rec_overall'],
            marker='*', c='blue', s=250, edgecolors='k', linewidths=1.0, zorder=5, label='Baseline')

# Annotate
for _, row in df_mc_pareto.iterrows():
    ax2.annotate(f't{int(row["trial_number"])}\n'
                 f'split={row["split_rate_rec_overall"]:.4f}',
                 (row['efficiency_true_overall'], row['fake_rate_rec_overall']),
                 textcoords='offset points', xytext=(6, 4), fontsize=7, alpha=0.8)

# Zoom to the front region
if len(df_mc_pareto) > 0:
    eff_margin = 0.02
    fake_margin = 0.005
    ax2.set_xlim(df_mc_pareto['efficiency_true_overall'].min() - eff_margin,
                 df_mc_pareto['efficiency_true_overall'].max() + eff_margin)
    ax2.set_ylim(df_mc_pareto['fake_rate_rec_overall'].min() - fake_margin,
                 df_mc_pareto['fake_rate_rec_overall'].max() + fake_margin)

sc2 = ax2.scatter([], [], c=[], cmap='YlOrRd')  # dummy for colorbar
ax2.set_xlabel('Efficiency (true) → higher better')
ax2.set_ylabel('Fake rate (rec) → lower better')
ax2.set_title(f'Zoomed: 2D Pareto Front (merge < {MERGE_CUT})\ncolored by split rate', fontsize=12)
ax2.legend(fontsize=9, loc='upper left')

fig.suptitle(f'Working Point Selection: merge rate < {MERGE_CUT}, then max eff & min fake',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Interpretation Notes

**Pareto front**:
- 322/610 trials are Pareto-optimal → the front is very broad in 4D, many trade-offs exist.
- The 2D projections reveal which objective pairs have strong trade-offs (anti-correlated front) vs which can be simultaneously improved.

**Parameter profiles**:
- A **flat profile** means the parameter has little effect on that objective.
- A **steep profile** with a clear optimum means the parameter is critical.
- When the **median** closely tracks the **envelope**, the parameter's effect is robust. When they diverge, the parameter interacts strongly with others.

**Working point selection**:
- **Utopia distance** gives the most "balanced" solution across all 4 objectives.
- **Constraint-based** (eff ≥ baseline, fake ≤ 2× baseline) narrows to physically acceptable points.
- **Knee point** identifies where the marginal cost of efficiency explodes — the natural operating point before diminishing returns.
- **Merge-constrained Pareto** (merge < 0.03) reduces the 4D problem to a 2D trade-off between efficiency and fake rate. The resulting 2D Pareto front shows exactly which (eff, fake) combinations are achievable while keeping merge under control. Each point on this front is a valid working point — the choice depends on whether you prioritize efficiency or purity.